# Regime-Shift: Macro-Aware Tactical Asset Allocation
FEC IIT Guwahati · DIY '26

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import cvxpy as cp
from scipy.stats import skew
import warnings
warnings.filterwarnings('ignore')

## 1. Data Pipeline

In [ ]:
ASSETS = ['SPY', 'TLT', 'GLD']
START  = '2005-01-01'
END    = '2024-12-31'

prices = yf.download(ASSETS, start=START, end=END, auto_adjust=True, progress=False)['Close']
vix    = yf.download('^VIX', start=START, end=END, auto_adjust=True, progress=False)['Close'].squeeze()
vix.name = 'VIX'

print(f'Downloaded {len(prices)} trading days')

In [ ]:
returns = np.log(prices / prices.shift(1)).dropna()
df = returns.join(vix, how='inner')

for col in ASSETS:
    df[col + '_vol21'] = df[col].rolling(21).std() * np.sqrt(252)

df.dropna(inplace=True)
print(f'Feature matrix: {df.shape}')
df.head()

In [ ]:
print(df[ASSETS].describe().round(4))

## 2. HMM Regime Classifier

In [ ]:
HMM_FEATURES = ['SPY', 'TLT', 'GLD', 'VIX', 'SPY_vol21', 'TLT_vol21', 'GLD_vol21']

scaler = StandardScaler()
X = scaler.fit_transform(df[HMM_FEATURES])

model = GaussianHMM(n_components=3, covariance_type='diag', n_iter=200, random_state=42)
model.fit(X)

print(f'Converged: {model.monitor_.converged}')
print(f'Log-likelihood: {model.monitor_.history[-1]:.2f}')

In [ ]:
states = model.predict(X)

means_orig = scaler.inverse_transform(model.means_)
means_df   = pd.DataFrame(means_orig, columns=HMM_FEATURES)

spy_means = means_df['SPY'].values
vix_means = means_df['VIX'].values
ranked    = np.argsort(spy_means)

labels = {}
labels[ranked[2]] = 'Bull'
if vix_means[ranked[0]] > vix_means[ranked[1]]:
    labels[ranked[0]] = 'Crisis'
    labels[ranked[1]] = 'Bear'
else:
    labels[ranked[1]] = 'Crisis'
    labels[ranked[0]] = 'Bear'

for state, name in labels.items():
    print(f'State {state}: {name}  SPY={spy_means[state]:.4f}  VIX={vix_means[state]:.2f}')

df['state']  = states
df['regime'] = df['state'].map(labels)
print('\nRegime distribution:')
print(df['regime'].value_counts())

In [ ]:
state_names = [labels[i] for i in range(3)]
tmat = pd.DataFrame(model.transmat_, index=state_names, columns=state_names)
print('Transition Matrix:')
print(tmat.round(3))

In [ ]:
REGIME_COLORS = {'Bull': '#2ecc71', 'Bear': '#e67e22', 'Crisis': '#e74c3c'}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

spy_cum = np.exp(df['SPY'].cumsum())
axes[0].plot(df.index, spy_cum, color='black', lw=1.2)
axes[0].set_ylabel('SPY cumulative return')
axes[0].set_title('HMM Regimes overlaid on SPY')
axes[0].grid(alpha=0.3)

axes[1].plot(df.index, df['VIX'], color='crimson', lw=0.9)
axes[1].axhline(30, color='gray', ls='--', lw=0.7)
axes[1].set_ylabel('VIX')
axes[1].grid(alpha=0.3)

prev_r = df['regime'].iloc[0]
start  = df.index[0]
for date, reg in df['regime'].items():
    if reg != prev_r:
        for ax in axes:
            ax.axvspan(start, date, alpha=0.15, color=REGIME_COLORS[prev_r])
        start = date
        prev_r = reg
for ax in axes:
    ax.axvspan(start, df.index[-1], alpha=0.15, color=REGIME_COLORS[prev_r])

patches = [mpatches.Patch(color=c, alpha=0.5, label=r) for r, c in REGIME_COLORS.items()]
axes[0].legend(handles=patches, loc='upper left')

plt.tight_layout()
plt.savefig('regime_chart.png', dpi=150)
plt.show()
print('saved regime_chart.png')

## 3. Walk-Forward Backtest

In [ ]:
TURNOVER_COST = 0.0010
TRAIN_DAYS    = 2 * 252
REBAL_FREQ    = 21
RISK_FREE     = 0.0001

asset_returns = df[ASSETS]
regimes       = df['regime']

def optimize(mu, sigma, prev_w, regime):
    w = cp.Variable(len(ASSETS))
    port_ret = mu @ w
    port_var = cp.quad_form(w, sigma)
    penalty  = TURNOVER_COST * cp.norm1(w - prev_w)

    constraints = [cp.sum(w) == 1, w >= 0.05, w <= 0.80]

    if regime == 'Bull':
        obj = cp.Maximize(port_ret - 0.5 * port_var - penalty)
    elif regime == 'Bear':
        constraints.append(port_var <= (0.12 / np.sqrt(252)) ** 2)
        obj = cp.Maximize(port_ret - 1.0 * port_var - penalty)
    else:
        obj = cp.Minimize(port_var + penalty)

    prob = cp.Problem(obj, constraints)
    prob.solve(solver=cp.SCS, verbose=False)

    if prob.status not in ['optimal', 'optimal_inaccurate'] or w.value is None:
        return prev_w

    out = np.clip(w.value, 0, 1)
    return out / out.sum()

In [ ]:
dates   = asset_returns.index
prev_w  = np.ones(len(ASSETS)) / len(ASSETS)
curr_w  = prev_w.copy()
results = []

print('Running backtest...')
for i in range(TRAIN_DAYS, len(dates)):
    if (i - TRAIN_DAYS) % REBAL_FREQ == 0:
        train_r = asset_returns.iloc[i - TRAIN_DAYS:i]
        mu      = train_r.mean().values * 252
        sigma   = train_r.cov().values  * 252
        curr_w  = optimize(mu, sigma, prev_w, regimes.iloc[i])
        prev_w  = curr_w.copy()

    results.append({
        'date':   dates[i],
        'return': asset_returns.iloc[i].values @ curr_w,
        'regime': regimes.iloc[i],
        'w_SPY':  curr_w[0],
        'w_TLT':  curr_w[1],
        'w_GLD':  curr_w[2],
    })

    if i % 500 == 0:
        print(f'  {i}/{len(dates)} days done')

res = pd.DataFrame(results).set_index('date')
print('Backtest complete.')

## 4. Performance

In [ ]:
bench = asset_returns.iloc[TRAIN_DAYS:].copy()
bench['60_40']  = bench['SPY'] * 0.6 + bench['TLT'] * 0.4
bench['eq_wt']  = bench[ASSETS].mean(axis=1)

def metrics(r, label):
    ann_ret  = r.mean() * 252
    ann_vol  = r.std()  * np.sqrt(252)
    sharpe   = (r.mean() - RISK_FREE) / r.std() * np.sqrt(252)
    downside = r[r < 0].std() * np.sqrt(252)
    sortino  = (ann_ret - RISK_FREE * 252) / downside
    cum      = (1 + r).cumprod()
    max_dd   = ((cum - cum.cummax()) / cum.cummax()).min()
    calmar   = ann_ret / abs(max_dd)
    print(f'{label}: Return={ann_ret:.2%} Sharpe={sharpe:.2f} MaxDD={max_dd:.2%} Calmar={calmar:.2f}')
    return dict(label=label, ann_ret=ann_ret, ann_vol=ann_vol,
                sharpe=sharpe, sortino=sortino, max_dd=max_dd, calmar=calmar)

m = []
m.append(metrics(res['return'],      'Regime-Shift'))
m.append(metrics(bench['60_40'],     '60/40'))
m.append(metrics(bench['eq_wt'],     'Equal Weight'))

summary = pd.DataFrame(m).set_index('label')
summary.round(3)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

strat_cum = (1 + res['return']).cumprod()
b6040_cum = (1 + bench['60_40']).cumprod()
ew_cum    = (1 + bench['eq_wt']).cumprod()

axes[0].plot(strat_cum.index, strat_cum, lw=1.5, label='Regime-Shift', color='#3498db')
axes[0].plot(b6040_cum.index, b6040_cum, lw=1.2, label='60/40',        color='#e67e22', ls='--')
axes[0].plot(ew_cum.index,    ew_cum,    lw=1.2, label='Equal Weight',  color='#9b59b6', ls='--')
axes[0].set_title('Equity Curves')
axes[0].set_ylabel('Growth of $1')
axes[0].legend()
axes[0].grid(alpha=0.3)

cum  = (1 + res['return']).cumprod()
dd   = (cum - cum.cummax()) / cum.cummax()
axes[1].fill_between(dd.index, dd, 0, color='#e74c3c', alpha=0.4)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown')
axes[1].grid(alpha=0.3)

axes[2].stackplot(res.index, res['w_SPY'], res['w_TLT'], res['w_GLD'],
                  labels=['SPY','TLT','GLD'], colors=['#3498db','#2ecc71','#f1c40f'], alpha=0.7)
axes[2].set_title('Portfolio Weights')
axes[2].set_ylabel('Weight')
axes[2].legend(loc='upper left')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('tearsheet.png', dpi=150)
plt.show()
print('saved tearsheet.png')

In [ ]:
res.to_csv('backtest_results.csv')
summary.to_csv('performance_summary.csv')
print('All files saved.')